<a href="https://colab.research.google.com/github/slr549/Machine-Learning-Course-2026/blob/main/assignments/week-10/2411070095_Raki%20Raihan/10_NLP_Sentiment_Analysis_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup & Import Data

In [1]:
import pandas as pd
import numpy as np
import re
import string
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import wandb

# 1. Inisialisasi W&B
run = wandb.init(project="nlp-sentiment-analysis", name="tfidf-naive-bayes-tfds")

# 2. Load Dataset IMDB dari TensorFlow Datasets (Sangat Stabil)
print("Sedang mengunduh dataset IMDB dari TensorFlow... (Mohon tunggu)")
ds, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)

# 3. Konversi ke Pandas DataFrame agar sesuai dengan alur notebook Anda
train_data = list(tfds.as_numpy(ds['train']))
df = pd.DataFrame(train_data, columns=['review', 'label'])

# Decode bytes ke string dan ubah label angka menjadi teks (0: neg, 1: pos)
df['review'] = df['review'].str.decode("utf-8")
df['label'] = df['label'].map({0: 'neg', 1: 'pos'})

# Ambil 10.000 sampel agar proses training tetap cepat
df = df.sample(10000, random_state=42)

print(f"Dataset Berhasil Dimuat! Jumlah data: {len(df)}")
df.head()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: han-dev321 (han-dev321-stikomelrahma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.S3YC9F_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.S3YC9F_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.
Dataset Berhasil Dimuat! Jumlah data: 10000


,review,label
6868,"I watched ""Elephant Walk"" for the first time i...",pos
24016,I would put this at the top of my list of film...,neg
9668,"Police, investigations, murder, suspicion: we ...",pos
13640,I read Schneebaum's book (same title as this f...,pos
14018,"Well, you'd better if you plan on sitting thro...",neg


Text Cleaning

In [2]:
def clean_text(text):
    text = text.lower() # Kecilkan semua
    text = re.sub('\[.*?\]', '', text) # Hapus teks dalam kurung
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # Hapus tanda baca
    text = re.sub('\w*\d\w*', '', text) # Hapus kata yang mengandung angka
    return text

df['review_clean'] = df['review'].apply(clean_text)
print("Contoh teks sebelum & sesudah dibersihkan:")
print(df[['review', 'review_clean']].iloc[0])

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_1953/1010002402.py:3: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text) # Hapus teks dalam kurung
/tmp/ipykernel_1953/1010002402.py:5: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text) # Hapus kata yang mengandung angka


Contoh teks sebelum & sesudah dibersihkan:
review          I watched "Elephant Walk" for the first time i...
review_clean    i watched elephant walk for the first time in ...
Name: 6868, dtype: object


Vectorization

In [3]:
# TF-IDF memberikan bobot lebih pada kata yang penting dan langka
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['review_clean'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

wandb.log({"vectorizer_type": "TF-IDF", "vocab_size": 5000})

Training & Evaluation

In [4]:
model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Akurasi Sentimen: {acc:.4f}")
print(classification_report(y_test, y_pred))

wandb.log({"Accuracy": acc})
wandb.finish()

Akurasi Sentimen: 0.8465
              precision    recall  f1-score   support

         neg       0.83      0.88      0.85      1010
         pos       0.87      0.82      0.84       990

    accuracy                           0.85      2000
   macro avg       0.85      0.85      0.85      2000
weighted avg       0.85      0.85      0.85      2000



Accuracy,▁
vocab_size,▁
Accuracy,0.8465
vectorizer_type,TF-IDF
vocab_size,5000
